In [ ]:
!python3 -m venv sailing-env
!source sailing-env/bin/activate
!pip install -r requirements.txt

In [ ]:
import sys
import os
import numpy as np

# Add the src directory to the path
sys.path.append(os.path.abspath('../src'))
sys.path.append(os.path.abspath('..'))

# Import the BaseAgent class
from src.agents.base_agent import BaseAgent

In [ ]:
from src.env_sailing import SailingEnv
from src.wind_scenarios import get_wind_scenario

In [ ]:
class FCNet(nn.Module): #on garde cette classe
  """
  Define the Neural network with the __init__ and forward method.
  It should define a fully connected
  neural network with prescribed input size, hidden size and output size
  """
  def __init__(self, input_size, hidden_size, output_size) -> None:
    super().__init__()
    self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

  def forward(self, x):
      return self.linear_relu_stack(x)


class NN(BaseEstimator):
  """
  A sklearn-like class for the neural net
  """
  def __init__(self, n_iterations, input_size, hidden_size, output_size, alpha, seed, batch_size) -> None:
    """
    Initialize the sklearn class:
    - Record the input parameters using
    self.parameter = parameter
    - Initialize the neural network model and record it in self.model
    - Initialize the Adam optimizer and record it in self.optimizer
    """

    super().__init__()
    self.n_iterations = n_iterations
    self.alpha = alpha
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size
    self.seed = seed
    self.batch_size = batch_size
    model = FCNet(input_size, hidden_size, output_size)
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
    self.model = model.to(device)
    optimizer = torch.optim.Adam(self.model.parameters(), lr=alpha)
    self.optimizer = optimizer


  def partial_fit(self, X, Y):
    """Update parameters
    - Convert numpy input data to torch
    - Define the loss
    - Find the gradient via automatic differentiation
    - Perform a gradient update of the parameters

    Parameters
    ----------
    X: np array of size (n, ds + da)
      state-action pairs
    Y: np array of size (n, 1)
      Estimate of the state action value function
    """
    Xs = X[:, :-1]
    Xa = X[:, -1]
    Xs = torch.from_numpy(Xs.copy()).to(torch.float)
    Xa = torch.from_numpy(Xa.copy()).to(torch.int)
    Y = torch.from_numpy(Y.copy()).to(torch.float)
    loss = ((self.model(Xs)[torch.arange(len(Xs)), Xa] - Y) ** 2).sum() / len(Xs)
    loss.backward()
    self.optimizer.step()
    self.optimizer.zero_grad()


  def fit(self, X, Y):
    """Applies n_iterations steps of gradient using function grad_step.
    At each iteration, all samples are shuffled and split into batches of size
    `batch_size`.
    The gradient steps are performed on each batch of data sequentially
    """
    idx_samples = np.arange(len(X))
    for _ in range(self.n_iterations):
      np.random.shuffle(idx_samples)
      for batch_slice in gen_batches(len(X), self.batch_size):
        Xb, Yb = X[idx_samples[batch_slice]].copy(), Y[idx_samples[batch_slice]].copy()
        self.partial_fit(Xb, Yb)

  def predict(self, X):
    """Use the fitted parameter to predict q(s, a)
    - Convert input data to Torch
    - Apply the model
    - Convert the output to numpy

    Parameters
    ---------
    X: np array of shape (n, ds + da)
    Return
    ------
    qSA: np array of shape (n,)
      qSA[i] is an estimate of q(s_i, a_i) computed with the pred function
      where s_i and a_i are given by X[i]
    """
    Xs = X[:, :-1]
    Xa = X[:, -1]
    with torch.no_grad():
      Xs = torch.from_numpy(Xs.copy()).to(torch.float)
      Xa = torch.from_numpy(Xa.copy()).to(torch.int)
      y = self.model(Xs)[torch.arange(len(Xs)), Xa]
    return y.numpy()

In [ ]:
from copy import deepcopy

def DQN(model, capacity, batch_size, C, eps, gamma, n_iterations):
  """ Implements DQN
  Initialization
  - policy to uniform
  - the buffer is an np array of size (capacity, 2 * len(s) + 3)
  - create a copy of the model with deepcopy(model) that will contain the target weights
  At iteration t
  - choose an action according to eps-greedy
  - record the data into the buffer
  - sample `batch_size` data points uniformly at random from the buffer
  - construct the dataset X, Y as in fitted Q iteration (Y is constructed using target weights)
  - fit the model (using partial_fit)
  - every C iterations set update target weights by setting old_model.w = model.w
  - update the policy

  Parameters
  -----------
  model: sklearn-like model
    The model for the q-value function
  capacity: int
    The capacity of the buffer
  batch_size: int
    The number of data points sampled from the buffer at each timestep
  C: int
    Number of iterations before the target_weights are updated
  eps: float
    epsilon in epsilon-greedy
  gamma: float
    discount factor
  n_iterations: int
    Number of iterations to perform

  Return
  ------
  policy: state -> action function
    The DQN policy
  """
  Qmax = None
  s0, _ = env.reset()
  s = s0.copy()
  n_actions = env.action_space.n
  pi = pi_uniform
  old_model = deepcopy(model)
  buffer = np.zeros((capacity, len(s) * 2 + 3))

  for t in tqdm(range(n_iterations)):
    if np.random.rand() < eps:
      a = np.random.randint(n_actions)
    else:
      a = pi(s)
    s2, r, term, trunc, _ = env.step(a)
    buffer[t % capacity, :] = np.array(s.copy().tolist() + [a, r, term] + s2.copy().tolist())
    if term or trunc:
        s, _ = env.reset()
    else:
      s = s2.copy()
    I = np.random.randint(min(t+1, capacity), size=batch_size)
    data = buffer[I]

    if t % C == 0:
      old_model = deepcopy(model)

    Xb = data[:, :len(s) + 1]
    Yb = data[:, len(s) + 1]
    # Make a step
    if Qmax is None:
        model.partial_fit(Xb, Yb)

    Qmax = np.max(
          [
              old_model.predict(np.column_stack([
                          data[:, len(s) + 3:],
                          np.ones(len(data)).reshape(-1, 1) * a
                          ]))
              for a in range(n_actions)
          ],axis=0)
    Yb = data[:, len(s) + 1] + gamma * (1 - data[:, len(s) + 2]) * Qmax
    model.partial_fit(Xb, Yb)

    def pi(s):
      q = [model.predict(np.array(s.tolist() + [a]).reshape(1, -1))[0]
      for a in range(n_actions)]
      return np.argmax(q)
  return pi

In [ ]:
capacity = 2000
batch_size = 200
eps = 0.5
gamma = 0.9
C = 20
model = NN(n_iterations=1,
           input_size=len(s0),
           hidden_size=24,
           output_size=env.action_space.n,
           alpha=0.001,
           batch_size=None,
           seed=0)
pi = DQN(model, capacity, batch_size, C, eps, gamma, n_iterations=10000)
print(monte_carlo_eval(env, pi, 1000, gamma=1))
play_policy(env, pi, 100)

In [ ]:
#le dqn que je suis en train d'écrire :
class DQNAgent(BaseAgent):
    def __init__(self, input_size=6, hidden_size=128, output_size=8 ,learning_rate=0.1, discount_factor=0.9, exploration_rate=0.1):
        super().__init__()
        self.np_random = np.random.default_rng()

        # Learning parameters
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.exploration_rate = exploration_rate   

        # Linear parameters
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2./input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2./hidden_size)
        self.b2 = np.zeros((1, hidden_size))  
        self.W3 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2./hidden_size)
        self.b3 = np.zeros((1, output_size))  

    def forward(self, X):
        self.l1 = np.dot(X, self.W1) + self.b1
        self.relu1 = np.maximum(0, self.l1) 
        self.l2 = np.dot(self.relu1, self.W2) + self.b2
        self.relu2 = np.maximum(0, self.l2) 
        self.l3 = np.dot(self.relu2, self.W3) + self.b3
        return self.l3



In [ ]:
class QLearningAgent(BaseAgent):
    """A simple Q-learning agent for the sailing environment using only local information."""
    
    def __init__(self, learning_rate=0.1, discount_factor=0.9, exploration_rate=0.1):
        super().__init__()
        self.np_random = np.random.default_rng()
        
        # Learning parameters
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.exploration_rate = exploration_rate
        
        # State discretization parameters
        self.position_bins = 8     # Discretize the grid into 64x64
        self.velocity_bins = 4     # Discretize velocity into 4 bins
        self.wind_bins = 8         # Discretize wind directions into 8 bins
        
        # Initialize Q-table
        # State space: position_x, position_y, velocity_direction, wind_direction
        # Action space: 9 possible actions
        self.q_table = {}
        
    def discretize_state(self, observation):
        """Convert continuous observation to discrete state for Q-table lookup."""
        # Extract position, velocity and wind from observation
        x, y = observation[0], observation[1]
        vx, vy = observation[2], observation[3]
        wx, wy = observation[4], observation[5]
        
        # Discretize position (assume 128x128 grid)
        grid_size = 128
        x_bin = min(int(x / grid_size * self.position_bins), self.position_bins - 1)
        y_bin = min(int(y / grid_size * self.position_bins), self.position_bins - 1)
        
        # Discretize velocity direction (ignoring magnitude for simplicity)
        v_magnitude = np.sqrt(vx**2 + vy**2)
        if v_magnitude < 0.1:  # If velocity is very small, consider it as a separate bin
            v_bin = 0
        else:
            v_direction = np.arctan2(vy, vx)  # Range: [-pi, pi]
            v_bin = int(((v_direction + np.pi) / (2 * np.pi) * (self.velocity_bins-1)) + 1) % self.velocity_bins
        
        # Discretize wind direction
        wind_direction = np.arctan2(wy, wx)  # Range: [-pi, pi]
        wind_bin = int(((wind_direction + np.pi) / (2 * np.pi) * self.wind_bins)) % self.wind_bins
        
        # Return discrete state tuple
        return (x_bin, y_bin, v_bin, wind_bin)
        
    def act(self, observation):
        """Choose an action using epsilon-greedy policy."""
        # Discretize the state
        state = self.discretize_state(observation)
        
        # Epsilon-greedy action selection
        if self.np_random.random() < self.exploration_rate:
            # Explore: choose a random action
            return self.np_random.integers(0, 9)
        else:
            # Exploit: choose the best action according to Q-table
            if state not in self.q_table:
                # If state not in Q-table, initialize it
                self.q_table[state] = np.zeros(9)
            
            # Return action with highest Q-value
            return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state):
        """Update Q-table based on observed transition."""
        # Initialize Q-values if states not in table
        if state not in self.q_table:
            self.q_table[state] = np.zeros(9)
        if next_state not in self.q_table:
            self.q_table[next_state] = np.zeros(9)
        
        # Q-learning update
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.discount_factor * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.learning_rate * td_error
    
    def reset(self):
        """Reset the agent for a new episode."""
        # Nothing to reset for Q-learning agent
        pass
        
    def seed(self, seed=None):
        """Set the random seed."""
        self.np_random = np.random.default_rng(seed)
        
    def save(self, path):
        """Save the Q-table to a file."""
        import pickle
        with open(path, 'wb') as f:
            pickle.dump(self.q_table, f)
            
    def load(self, path):
        """Load the Q-table from a file."""
        import pickle
        with open(path, 'rb') as f:
            self.q_table = pickle.load(f)

In [ ]:
import math
# l_1 = [0,1,0.1,0.01]
# l_2 = [0,1,0.1,0.01]
# l_3 = [0,1,0.1,0.01]
# l_4 = [0,1,0.1,0.01]
c_1 = 0.5
c_2=0.1
c_3=0.05
c_4 = 0.5
# Full training across all 3 wind scenarios with reward shaping
ql_agent_full = QLearningAgent(learning_rate=0.1, discount_factor=0.995, exploration_rate=0.5)
max_steps = 500

np.random.seed(42)
ql_agent_full.seed(42)

scenarios = ['training_1', 'training_2', 'training_3']
num_episodes = 2000

rewards_history = []
steps_history = []
success_history = []

import time
start_time = time.time()
print(f"Starting full training with {num_episodes} episodes across 3 scenarios...")

for episode in range(num_episodes):
    scenario = scenarios[episode % 3]
    env = SailingEnv(**get_wind_scenario(scenario))
    goal = env.goal_position.copy()
    
    observation, info = env.reset(seed=episode)
    state = ql_agent_full.discretize_state(observation)
    prev_dist = np.linalg.norm(info['position'] - goal)
    total_reward = 0
    
    for step in range(max_steps):
        action = ql_agent_full.act(observation)
        next_observation, reward, done, truncated, info = env.step(action)
        next_state = ql_agent_full.discretize_state(next_observation)
        
        #Reward 

        #VMG reward
        curr_speed = math.sqrt((info['velocity'][0])**2 + (info['velocity'][1])**2 )
        curr_angle_boat = math.atan2(info['velocity'][1], info['velocity'][0])
        curr_angle_to_goal = math.atan2(goal[1] - info['position'][1], goal[0] - info['position'][0])
        theta = curr_angle_boat - curr_angle_to_goal
        vmg_reward = curr_speed * math.cos(theta)

        #angle wind reward 
        curr_angle_wind = math.degrees(math.atan2(info['wind'][1], info['wind'][0]))
        curr_angle_boat = math.degrees(math.atan2(info['velocity'][1], info['velocity'][0]))
        curr_angle = abs(curr_angle_wind - curr_angle_boat)
        if curr_angle > 180 :
            curr_angle = 360 - curr_angle
        curr_angle = 180 - curr_angle
        if curr_angle < 40:
            wind_reward = -1.0
        elif curr_angle < 50:
            wind_reward = 0.4
        elif curr_angle < 80:
            wind_reward = 0.6
        elif curr_angle < 100:
            wind_reward = 1.0
        elif curr_angle < 150:
            wind_reward = 0.8
        else:
            wind_reward = 0.4
        
        #speed reward 
        speed_reward = -0.1

        #change action reward 
        action_reward = 0
        if previous_action != None and action != previous_action :
            action_reward = -0.2

        #distance reward
        curr_dist = np.linalg.norm(info['position'] - goal) 
        distance_reward = prev_dist - ql_agent_full.discount_factor*curr_dist

        if step == 50 and (episode+1)%500 == 0:
            print(f"Distance: {distance_reward:.4f}")
            print(f"VMG: {vmg_reward:.4f}")
            print(f"Wind: {wind_reward:.4f}")
            print(f"Action Pen: {action_reward:.4f}")
        shaped_reward = reward + 20*speed_reward + 0*distance_reward + 4*vmg_reward + 0.5*wind_reward + action_reward
        if step == 50 and (episode+1)%500 == 0:
            print(f"TOTAL STEP REWARD: {shaped_reward:.4f}")

        #shaped_reward = reward + c_1*distance_reward + c_2*vmg_reward + c_3*wind_reward + c_4*speed_reward 

        if done and not info.get('is_stuck', False):
            shaped_reward += (max_steps - step) * 0.5
        if info.get('is_stuck', False):
            shaped_reward = -100.0
        prev_dist = curr_dist
        
        ql_agent_full.learn(state, action, shaped_reward, next_state)
        state = next_state
        observation = next_observation
        total_reward += reward
        
        if done or truncated:
            break

    rewards_history.append(total_reward)
    steps_history.append(step + 1)
    success_history.append(done)
    ql_agent_full.exploration_rate = max(0.05, ql_agent_full.exploration_rate * 0.998)
    
    if (episode + 1) % 500 == 0:
        recent = success_history[-100:]
        print(f"Episode {episode+1}/{num_episodes}: Success rate (last 100): {sum(recent)/len(recent)*100:.1f}%")

training_time = time.time() - start_time
success_rate = sum(success_history) / len(success_history) * 100
print(f"\nTraining completed in {training_time:.1f} seconds!")
print(f"Overall success rate: {success_rate:.1f}%")
print(f"Q-table size: {len(ql_agent_full.q_table)} states")

q_table_save_path = "trained_test.pkl"
ql_agent_full.save(q_table_save_path)